In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.nn.functional as F
import matplotlib.pyplot as plt
import get_dataset
import cnn
import transformer
import ff
import transformers.models.patchtst.modeling_patchtst as ptst_modeling
import transformers.models.patchtst.configuration_patchtst as ptst_config
import importlib
from einops import rearrange
from bayes_opt import BayesianOptimization
from matplotlib.ticker import MaxNLocator
from time import time
import os

torch.manual_seed(42)

c:\Users\User\anaconda3\envs\clean\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PATH_D = "Series temporales/DNI/"
NUM_S = 3
SENSORS = ["S1", "S3", "S4"]

In [3]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device = "cpu"
print(f"Using {device} device")

Using cpu device


In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):

    model.train()
    patchtst = model.name == "PatchTST"

    for batch, (X, y) in enumerate(dataloader):
        if patchtst:
            X = rearrange(X, 'b c s -> b s c')

        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()


def test_loop(dataloader, model, loss_fn, threshold, verbose=True):

    model.eval()
    patchtst = model.name == "PatchTST"
    num_batches = len(dataloader)
    positive_real, positive_pred, test_loss, correct, correct_pos = 0, 0, 0, 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            if patchtst:
                X = rearrange(X, 'b c s -> b s c')
 
            pred = model(X)
            loss = loss_fn(pred, y).item()
            
            test_loss += loss  

            pred_pos = torch.where(F.sigmoid(pred) > threshold, 1, 0)
            real_pos = (y == 1.0)

            positive_pred += pred_pos.type(torch.float).sum().item()
            positive_real += y.type(torch.float).sum().item()
            correct_pos += torch.bitwise_and(pred_pos,real_pos).type(torch.float).sum().item()
            correct += (pred_pos == y).type(torch.float).sum().item()


    test_loss /= num_batches
    correct /= 3*num_batches*dataloader.batch_size
    if (positive_pred != 0):
        precission = correct_pos/positive_pred
    else:
        precission = 0.0
    if (positive_real != 0):
        recall = correct_pos/positive_real
    else:
        recall = 0.0
    if (precission+recall > 0):
        f1score = 2*precission*recall/(precission+recall)
    else:
        f1score = 0

    if (verbose):
        print(f"Accuracy: \t {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f}")
        print(f"Precission: \t {(100*precission):>0.1f}%")
        print(f"Recall: \t {(100*recall):>0.1f}%")
        print(f"F1-score: \t {(f1score):>0.3f}")

    return test_loss, correct, precission, recall, f1score


In [5]:
batch_size = 16
memory = 1
importlib.reload(get_dataset)
train_ds, test_ds, val_ds, weights = get_dataset.get_dataset(0.20,0.20,sequential=False,coef_weights=1,random_state=42,device=device)

Categorías entrenamiento:
[0 1 2 3 4 5 6 7]
[855 143 186   3 203   7  17   8]
Categorías validación:
[0 1 2 3 4 5 6 7]
[285  47  62   1  67   3   6   3]
Categorías test:
[0 1 2 3 4 5 6 7]
[286  48  62   1  68   2   6   2]


In [6]:
train_loader = DataLoader(train_ds,batch_size=batch_size,shuffle=False,drop_last=True)
test_loader = DataLoader(test_ds,batch_size=batch_size,shuffle=False,drop_last=True)
val_loader = DataLoader(val_ds,batch_size=batch_size,shuffle=False,drop_last=True)

In [7]:
def train_loop_optimize(model, criterion, optimizer, threshold,min_recall):
    epochs = 80

    best_epoch = 0
    best_metrics = (99,0,0,0,0)

    for epoch in range(epochs):
        train_loop(train_loader, model, criterion, optimizer)
        metrics = test_loop(val_loader,model,criterion,threshold=threshold,verbose=False)

        if (metrics[-2] >= min_recall):
            if (metrics[-1] > best_metrics[-1]):
                best_metrics = metrics
                best_epoch = epoch

    return best_epoch, *best_metrics

In [8]:
def print_metrics(best_score, best_epoch, loss, accuracy, precission, recall):
    print(f"Accuracy: \t {(100*accuracy):>0.1f}%")
    print(f"Precission: \t {(100*precission):>0.1f}%")
    print(f"Recall: \t {(100*recall):>0.1f}%")
    print(f"F1-score: \t {(best_score):>0.4f}")
    print(f"Loss: \t {(loss):>0.4f}")
    print(f"Mejor epoch: \t {(best_epoch)}")

In [9]:
def train_ff(learning_rate,threshold,layer1,layer2):
    criterion = nn.BCEWithLogitsLoss(pos_weight=weights.to(device))
    model = ff.FF(num_sensors=NUM_S,layer1=layer1,layer2=layer2).to(device)
    optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)

    best_epoch, loss, accuracy, precission, recall, best_score = train_loop_optimize(model, criterion, optimizer, threshold, 0.8)

    print_metrics(best_score, best_epoch, loss, accuracy, precission, recall)

    return best_score


def train_cnn(learning_rate,threshold,channel1,channel2,channel3,kernel1,kernel2,kernel3,dense1,dense2):
    criterion = nn.BCEWithLogitsLoss(pos_weight=weights.to(device))
    channels = [channel1,channel2,channel3]
    kernels = [kernel1,kernel2,kernel3]
    dense = [dense1, dense2]
    model = cnn.CNN(num_sensors=NUM_S,batch_size=batch_size,channels=channels,kernels=kernels,dense=dense).to(device)
    optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)

    best_epoch, loss, accuracy, precission, recall, best_score = train_loop_optimize(model, criterion, optimizer, threshold, 0.85)

    print_metrics(best_score, best_epoch, loss, accuracy, precission, recall)

    return best_score


def train_vit(learning_rate, threshold,dim,depth,heads,dim_head,mlp_dim,dropout,emb_dropout):
    criterion = nn.BCEWithLogitsLoss(pos_weight=weights.to(device))
    model = transformer.ViT(series_length=144,
                            patch_length=6,
                            num_sensors=NUM_S,
                            dim=dim,
                            depth=depth,
                            heads=heads,
                            dim_head=dim_head,
                            mlp_dim=mlp_dim,
                            dropout=dropout,
                            emb_dropout=emb_dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)

    best_epoch, loss, accuracy, precission, recall, best_score = train_loop_optimize(model, criterion, optimizer, threshold, 0.85)
    print_metrics(best_score, best_epoch, loss, accuracy, precission, recall)

    return best_score



def train_ptst(learning_rate, threshold, ffn_dim, num_hidden_layers, num_attention_heads, attention_dropout, positional_dropout):
    criterion = nn.BCEWithLogitsLoss(pos_weight=weights.to(device))
    config = ptst_config.PatchTSTConfig(
        num_input_channels=NUM_S,
        num_targets=NUM_S,
        d_model=12,
        ffn_dim=ffn_dim,
        num_hidden_layers=num_hidden_layers,
        num_attention_heads=num_attention_heads,
        attention_dropout=attention_dropout,
        positional_dropout=positional_dropout,
        context_length=144,
        patch_length=12,
        patch_stride=12,
        use_cls_token=False,
        channel_attention=True
    )
    model = ptst_modeling.PatchTSTForClassification(config=config).to(device)
    optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)

    best_epoch, loss, accuracy, precission, recall, best_score = train_loop_optimize(model, criterion, optimizer, threshold, 0.85)
    print_metrics(best_score, best_epoch, loss, accuracy, precission, recall)

    return best_score

In [10]:
pbounds_ff = {'learning_rate': (0.0001, 0.001), 'threshold': (0,1), 'layer1': (10,1000, int), 'layer2':(10,1000,int)}
pbounds_cnn = {'learning_rate': (0.0001, 0.001), 
               'threshold': (0,1), 
               'channel1': (1, 5, int), 
               'channel2': (1, 5, int), 
               'channel3': (1, 5, int), 
               'kernel1':(3,5,int),
               'kernel2':(3,5,int),
               'kernel3':(3,5,int),
               'dense1':(10,500,int),
               'dense2':(10,200,int)}

pbounds_vit = {'learning_rate': (0.0001, 0.001), 
               'threshold': (0,1),
               'dim': (6,18,int),
               'depth': (2,4,int),
               'heads': (2,4,int),
               'dim_head': (4,12,int),
               'mlp_dim': (6,200,int),
               'dropout':(0,0.2),
               'emb_dropout':(0,0.2)
}

pbounds_ptst = {'learning_rate': (0.0001,0.001),
               'threshold': (0,1),
               'ffn_dim': (50,200,int),
               'num_attention_heads': (2,4,int),
               'num_hidden_layers': (2,4,int),
               'attention_dropout': (0,0.2),
               'positional_dropout': (0,0.2)
               }


In [13]:
INIT_POINTS = 1
N_ITER = 2

In [14]:
optimizer_ff = BayesianOptimization(
    f=train_ff,
    pbounds=pbounds_ff,
    verbose=2, # verbose = 1 prints only when a maximum is observed, verbose = 0 is silent
    random_state=42,
)

optimizer_ff.maximize(
    init_points = INIT_POINTS,
    n_iter = N_ITER
)

C:\Users\User\AppData\Local\Temp\ipykernel_16560\2205255645.py:1: UserWarning: Non-float parameters are experimental and may not work as expected. Exercise caution when using them and please report any issues you encounter.
  optimizer_ff = BayesianOptimization(


|   iter    |  target   | learni... | threshold |  layer1   |  layer2   |
-------------------------------------------------------------------------
Accuracy: 	 0.0%
Precission: 	 0.0%
Recall: 	 0.0%
F1-score: 	 0.0000
Loss: 	 99.0000
Mejor epoch: 	 0
| 2         | 0.0       | 0.0004370 | 0.9507143 | 116       | 81        |


ValueError: too many values to unpack (expected 6)

In [ ]:
optimizer_cnn = BayesianOptimization(
    f=train_cnn,
    pbounds=pbounds_cnn,
    verbose=2, # verbose = 1 prints only when a maximum is observed, verbose = 0 is silent
    random_state=42,
)

optimizer_cnn.maximize(
    init_points = INIT_POINTS,
    n_iter = N_ITER
)

|   iter    |  target   | learni... | threshold | channel1  | channel2  | channel3  |  kernel1  |  kernel2  |  kernel3  |  dense1   |  dense2   |
-------------------------------------------------------------------------------------------------------------------------------------------------


C:\Users\User\AppData\Local\Temp\ipykernel_22704\1584754282.py:1: UserWarning: Non-float parameters are experimental and may not work as expected. Exercise caution when using them and please report any issues you encounter.
  optimizer_cnn = BayesianOptimization(


Accuracy: 	 94.0%
Precission: 	 86.7%
Recall: 	 85.0%
F1-score: 	 0.8586
Loss: 	 0.8761
Mejor epoch: 	 36
| 2         | 0.8585858 | 0.0004370 | 0.9507143 | 3         | 5         | 5         | 5         | 4         | 5         | 224       | 84        |
Accuracy: 	 94.9%
Precission: 	 92.6%
Recall: 	 86.1%
F1-score: 	 0.8923
Loss: 	 1.6122
Mejor epoch: 	 56
| 3         | 0.8923076 | 0.0005133 | 0.3337086 | 3         | 5         | 2         | 4         | 4         | 4         | 201       | 197       |
Accuracy: 	 92.2%
Precission: 	 75.9%
Recall: 	 87.6%
F1-score: 	 0.8129
Loss: 	 0.8853
Mejor epoch: 	 29
| 4         | 0.8129330 | 0.0002650 | 0.3042422 | 5         | 4         | 1         | 3         | 5         | 5         | 179       | 197       |
Accuracy: 	 94.2%
Precission: 	 88.8%
Recall: 	 85.3%
F1-score: 	 0.8700
Loss: 	 0.8595
Mejor epoch: 	 9
| 5         | 0.87      | 0.0004297 | 0.4560699 | 3         | 4         | 4         | 3         | 5         | 3         | 316       | 144  

In [ ]:
optimizer_vit = BayesianOptimization(
    f=train_vit,
    pbounds=pbounds_vit,
    verbose=2, # verbose = 1 prints only when a maximum is observed, verbose = 0 is silent
    random_state=42,
)

optimizer_vit.maximize(
    init_points = INIT_POINTS,
    n_iter = N_ITER
)

|   iter    |  target   | learni... | threshold |    dim    |   depth   |   heads   | dim_head  |  mlp_dim  |  dropout  | emb_dr... |
-------------------------------------------------------------------------------------------------------------------------------------


C:\Users\User\AppData\Local\Temp\ipykernel_22704\287294197.py:1: UserWarning: Non-float parameters are experimental and may not work as expected. Exercise caution when using them and please report any issues you encounter.
  optimizer_vit = BayesianOptimization(


Accuracy: 	 0.0%
Precission: 	 0.0%
Recall: 	 0.0%
F1-score: 	 0.0000
Loss: 	 99.0000
Mejor epoch: 	 0
| 2         | 0.0       | 0.0004370 | 0.9507143 | 16        | 2         | 2         | 10        | 127       | 0.0311989 | 0.0116167 |
Accuracy: 	 92.8%
Precission: 	 79.4%
Recall: 	 85.4%
F1-score: 	 0.8232
Loss: 	 0.4735
Mejor epoch: 	 70
| 3         | 0.8232445 | 0.0008795 | 0.6011150 | 13        | 4         | 3         | 8         | 7         | 0.1443997 | 0.1877105 |
Accuracy: 	 0.0%
Precission: 	 0.0%
Recall: 	 0.0%
F1-score: 	 0.0000
Loss: 	 99.0000
Mejor epoch: 	 0
| 4         | 0.0       | 0.0001007 | 0.9922115 | 6         | 3         | 3         | 12        | 54        | 0.1049549 | 0.0799721 |
Accuracy: 	 0.0%
Precission: 	 0.0%
Recall: 	 0.0%
F1-score: 	 0.0000
Loss: 	 99.0000
Mejor epoch: 	 0
| 5         | 0.0       | 0.0001419 | 0.9737555 | 8         | 4         | 2         | 6         | 56        | 0.1360615 | 0.0900998 |
Accuracy: 	 0.0%
Precission: 	 0.0%
Recall: 	 0.0

In [ ]:
optimizer_ptst = BayesianOptimization(
    f=train_ptst,
    pbounds=pbounds_ptst,
    verbose=2, # verbose = 1 prints only when a maximum is observed, verbose = 0 is silent
    random_state=42,
)

optimizer_ptst.maximize(
    init_points = INIT_POINTS,
    n_iter = N_ITER
)

C:\Users\User\AppData\Local\Temp\ipykernel_17744\662716395.py:1: UserWarning: Non-float parameters are experimental and may not work as expected. Exercise caution when using them and please report any issues you encounter.
  optimizer_ptst = BayesianOptimization(


|   iter    |  target   | learni... | threshold |  ffn_dim  | num_at... | num_hi... | attent... | positi... |
-------------------------------------------------------------------------------------------------------------
Accuracy: 	 0.0%
Precission: 	 0.0%
Recall: 	 0.0%
F1-score: 	 0.0000
Loss: 	 99.0000
Mejor epoch: 	 0
| 2         | 0.0       | 0.0004370 | 0.9507143 | 156       | 2         | 2         | 0.0312037 | 0.0311989 |
Accuracy: 	 0.0%
Precission: 	 0.0%
Recall: 	 0.0%
F1-score: 	 0.0000
Loss: 	 99.0000
Mejor epoch: 	 0
| 3         | 0.0       | 0.0001522 | 0.8661761 | 149       | 4         | 3         | 0.0112823 | 0.1443997 |
Accuracy: 	 28.5%
Precission: 	 16.4%
Recall: 	 98.0%
F1-score: 	 0.2811
Loss: 	 1.6106
Mejor epoch: 	 0
| 4         | 0.2811361 | 0.0009446 | 0.0007787 | 70        | 2         | 3         | 0.1049512 | 0.0863890 |
Accuracy: 	 0.0%
Precission: 	 0.0%
Recall: 	 0.0%
F1-score: 	 0.0000
Loss: 	 99.0000
Mejor epoch: 	 0
| 5         | 0.0       | 0.0003621 

In [ ]:
print(optimizer_ff.max)
print(optimizer_cnn.max)
print(optimizer_vit.max)
print(optimizer_ptst.max)

NameError: name 'optimizer_ff' is not defined